# Notebook 13 — Evaluate RAG Explanations

## Goal
Create a structured manual evaluation template for the generated explanations.
Compute summary statistics from any completed evaluations.

**Manual evaluation is required.** You must read each explanation and score it yourself.
This notebook creates the template; you fill it in, then rerun the summary cell.

---

## Scoring rubric

| Field | Scale | Meaning |
|-------|-------|---------|
| `correct_time_window_0_1` | 0 or 1 | 1 = all cited articles are dated before forecast_date |
| `labor_relevance_0_1` | 0 or 1 | 1 = articles are genuinely about the US labor market |
| `citation_support_0_1` | 0 or 1 | 1 = cited articles actually support the claims made |
| `faithfulness_1_5` | 1–5 | 5 = no invented facts; 1 = multiple unsupported claims |
| `usefulness_1_5` | 1–5 | 5 = clearly helps interpret the forecast; 1 = generic/unhelpful |

---

## What can go wrong
- Explanations generated from template (no API) will score high on faithfulness but low on usefulness
- DeepSeek may cite article titles not in the evidence (hallucination)
- COVID-period explanations may be vacuous (everyone knows COVID caused layoffs)

---

## What you must inspect
- Read every explanation before scoring
- Flag any citation that appears fabricated (title not found in retrieved_evidence.parquet)
- Note patterns in what makes explanations useful vs. useless

In [ ]:
import pathlib, json, datetime
import pandas as pd
import numpy as np

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/labor_news_rag_project')
REPO_DIR = pathlib.Path('/content/economic-news-labor-rag')

EXPLANATIONS_PATH = DRIVE_ROOT / 'outputs' / 'explanations' / 'forecast_explanations.parquet'
EVIDENCE_PATH = DRIVE_ROOT / 'outputs' / 'explanations' / 'retrieved_evidence.parquet'
TABLES_DIR = DRIVE_ROOT / 'outputs' / 'tables'

ap_path = DRIVE_ROOT / 'approvals' / '12_deepseek_explanations_approved.json'
if not ap_path.exists():
    raise FileNotFoundError('STOP: Notebook 12 not approved.')
with open(ap_path) as f:
    ap = json.load(f)
assert ap['approved'], 'NB 12 not approved.'

explanations_df = pd.read_parquet(EXPLANATIONS_PATH)
evidence_df = pd.read_parquet(EVIDENCE_PATH)
evidence_df['forecast_date'] = pd.to_datetime(evidence_df['forecast_date'])
evidence_df['article_date'] = pd.to_datetime(evidence_df['article_date'])

print(f'Explanations loaded: {len(explanations_df)}')
print(f'Evidence loaded: {len(evidence_df)}')

## Review all explanations with their evidence

In [ ]:
def print_explanation_for_review(exp_row, evidence_df):
    fd = pd.Timestamp(exp_row['forecast_date'])
    ev = evidence_df[evidence_df['forecast_date'] == fd]

    print(f'\n{"="*70}')
    print(f'FORECAST MONTH : {exp_row["forecast_month"]}')
    print(f'PREDICTION     : {exp_row["prediction"]:+.0f}K')
    print(f'ACTUAL         : {exp_row["actual"]:+.0f}K')
    print(f'ERROR          : {exp_row["actual"] - exp_row["prediction"]:+.0f}K')
    print(f'METHOD         : {exp_row["explanation_method"]}')
    print(f'\nRECOVERED EVIDENCE ({len(ev)} articles):')
    for i, er in enumerate(ev.itertuples(), 1):
        print(f'  {i}. [{er.query_group}] {er.article_date.date()} — {er.title[:70]}')
    print(f'\nEXPLANATION:')
    print(exp_row['explanation'])
    print()

print('Showing all explanations for manual review...')
print('(Read each one carefully before filling in the evaluation template below)')

for _, row in explanations_df.iterrows():
    print_explanation_for_review(row, evidence_df)

## Create manual evaluation template

This cell creates the evaluation template CSV. **You must open it, fill in the scores manually,
and re-upload it to Google Drive** before running the summary cell below.

In [ ]:
template_rows = []
for _, row in explanations_df.iterrows():
    fd = pd.Timestamp(row['forecast_date'])
    ev = evidence_df[evidence_df['forecast_date'] == fd]
    cited_urls = json.loads(row['cited_urls']) if isinstance(row['cited_urls'], str) else []

    template_rows.append({
        'forecast_month': row['forecast_month'],
        'prediction': round(row['prediction'], 1),
        'actual': round(row['actual'], 1),
        'explanation_method': row['explanation_method'],
        'n_articles_retrieved': row['n_articles_retrieved'],
        'explanation': row['explanation'][:500] + '...' if len(str(row['explanation'])) > 500 else row['explanation'],
        'cited_urls': ' | '.join(cited_urls[:3]),
        # Evaluation fields — FILL THESE IN MANUALLY
        'correct_time_window_0_1': '',
        'labor_relevance_0_1': '',
        'citation_support_0_1': '',
        'faithfulness_1_5': '',
        'usefulness_1_5': '',
        'notes': '',
    })

template_df = pd.DataFrame(template_rows)
template_path = TABLES_DIR / 'rag_evaluation_template.csv'
template_df.to_csv(template_path, index=False)

print(f'Evaluation template saved: {template_path}')
print(f'Rows to evaluate: {len(template_df)}')
print()
print('INSTRUCTIONS:')
print('1. Download rag_evaluation_template.csv from Google Drive')
print('2. Fill in: correct_time_window_0_1, labor_relevance_0_1,')
print('   citation_support_0_1, faithfulness_1_5, usefulness_1_5, notes')
print('3. Re-upload as rag_evaluation_completed.csv to the same folder')
print('4. Run the summary cells below')

## Summary statistics from completed evaluation

Run this cell only after you have completed and uploaded `rag_evaluation_completed.csv`.

In [ ]:
completed_path = TABLES_DIR / 'rag_evaluation_completed.csv'

if not completed_path.exists():
    print('Completed evaluation file not found.')
    print(f'Expected at: {completed_path}')
    print('Fill in the template and upload it before running this cell.')
else:
    completed_df = pd.read_csv(completed_path)
    score_cols = ['correct_time_window_0_1', 'labor_relevance_0_1',
                  'citation_support_0_1', 'faithfulness_1_5', 'usefulness_1_5']

    for col in score_cols:
        completed_df[col] = pd.to_numeric(completed_df[col], errors='coerce')

    print('=== RAG Evaluation Summary ===')
    summary = completed_df[score_cols].agg(['mean', 'std', 'min', 'max']).round(3)
    print(summary.to_string())
    print()
    print(f'Evaluated: {completed_df[score_cols[0]].notna().sum()} / {len(completed_df)} explanations')

    summary_path = TABLES_DIR / 'rag_evaluation_summary.csv'
    summary.to_csv(summary_path)
    print(f'Summary saved: {summary_path}')

## Hallucination check: are cited titles in the evidence?

In [ ]:
print('=== Hallucination Check ===')
print('Checking whether cited URLs in explanations were actually in the retrieved evidence...')
print()

all_evidence_urls = set(evidence_df['url'].str.strip().tolist())

hallucination_report = []
for _, row in explanations_df.iterrows():
    cited = json.loads(row['cited_urls']) if isinstance(row['cited_urls'], str) else []
    for url in cited:
        in_evidence = url.strip() in all_evidence_urls
        if not in_evidence:
            hallucination_report.append({
                'forecast_month': row['forecast_month'],
                'cited_url': url[:80],
                'in_evidence': False,
            })

if hallucination_report:
    hal_df = pd.DataFrame(hallucination_report)
    print(f'WARNING: {len(hal_df)} cited URLs are NOT in the retrieved evidence!')
    print('These may be hallucinated by DeepSeek. Review manually.')
    print(hal_df.to_string(index=False))
else:
    print('All cited URLs found in retrieved evidence. No hallucination detected.')

## Manual Approval Gate

**Before approving:**
1. You have read all explanations
2. Hallucination check passed (or flagged cases documented)
3. Evaluation template created and ready for manual scoring
4. At least a sample of explanations scored

In [ ]:
APPROVE_THIS_STEP = False

if not APPROVE_THIS_STEP:
    raise RuntimeError(
        'STOP: Inspect the diagnostics above. '
        'If everything is correct, change APPROVE_THIS_STEP=True and rerun this cell.'
    )

In [ ]:
approval = {
    'approved': True,
    'notebook': '13_evaluate_rag_explanations.ipynb',
    'approved_at': datetime.datetime.utcnow().isoformat(),
    'input_artifacts': [str(EXPLANATIONS_PATH), str(EVIDENCE_PATH)],
    'output_artifacts': [str(template_path)],
    'row_counts': {'evaluation_template_rows': int(len(template_df))},
    'warnings': [f'{len(hallucination_report)} URLs not in evidence'] if hallucination_report else [],
    'notes': ''
}
ap_path = DRIVE_ROOT / 'approvals' / '13_rag_evaluation_approved.json'
with open(ap_path, 'w') as f:
    json.dump(approval, f, indent=2)
print(f'Approval saved: {ap_path}')
print('Notebook 13 complete. Proceed to 14_create_paper_tables_and_figures.ipynb')